# 💾 Data Engineering & SQL Handbook

Этот ноутбук посвящен работе с данными: от парсинга сложных JSON-структур до продвинутых оконных функций в SQL.

---

## Блок 1: Работа с JSON и JSON Lines

### 🔎 Теория
- **JSON:** Стандарт для API. Бывает вложенным.
- **JSON Lines (`.jsonl`):** Каждая строка — отдельный JSON объект. Удобно для очень больших файлов.
- **`pd.json_normalize()`:** Функция Pandas, которая превращает вложенные словари в плоскую таблицу.

In [1]:
import pandas as pd

# Пример вложенного JSON
nested_data = [
    {'id': 1, 'info': {'name': 'Ivan', 'city': 'Moscow'}},
    {'id': 2, 'info': {'name': 'Anna', 'city': 'SPb'}}
]

# Превращаем в плоскую таблицу
df = pd.json_normalize(nested_data)
print(df)

   id info.name info.city
0   1      Ivan    Moscow
1   2      Anna       SPb


--- 
## Блок 2: SQL Sandbox (SQLite3)

Здесь мы создадим базу данных прямо в оперативной памяти.

In [ ]:
import sqlite3

# Создаем соединение с БД в памяти
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Создаем таблицу пользователей и заказов
cursor.execute('CREATE TABLE users (id INT, name TEXT, age INT)')
cursor.execute('CREATE TABLE orders (id INT, user_id INT, amount FLOAT)')

cursor.executemany('INSERT INTO users VALUES (?,?,?)', [(1, 'Ivan', 25), (2, 'Anna', 30), (3, 'Oleg', 22)])
cursor.executemany('INSERT INTO orders VALUES (?,?,?)', [(101, 1, 500.5), (102, 1, 1200.0), (103, 2, 350.0)])

conn.commit()

def run_sql(query):
    return pd.read_sql_query(query, conn)

print("База данных готова!")

--- 
## Блок 3: Агрегаты, GROUP BY и HAVING

**HAVING** используется для фильтрации **после** группировки (в отличие от WHERE).

In [ ]:
# Найти пользователей, у которых сумма заказов > 1000
query = """
SELECT user_id, SUM(amount) as total
FROM orders
GROUP BY user_id
HAVING total > 1000
"""
run_sql(query)

--- 
## Блок 4: Виды JOIN

- **INNER JOIN:** Только те, кто есть в обеих таблицах.
- **LEFT JOIN:** Все из левой + совпадения из правой (если нет — NULL).
- **CROSS JOIN:** Каждый с каждым (декартово произведение).

In [ ]:
# Все пользователи и их заказы (включая тех, у кого нет заказов)
query = """
SELECT u.name, o.amount
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
"""
run_sql(query)

--- 
## Блок 5: Оконные функции (Window Functions)

Позволяют делать вычисления по группе строк, не схлопывая их в одну (как GROUP BY).

- `ROW_NUMBER()` — порядковый номер внутри группы.
- `RANK()` — ранг (одинаковые значения получают один номер).
- `LAG() / LEAD()` — доступ к предыдущей / следующей строке.

In [ ]:
# Пронумеровать заказы для каждого пользователя по убыванию суммы
query = """
SELECT user_id, amount,
       ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY amount DESC) as row_num
FROM orders
"""
run_sql(query)

### 📝 Практические задачи

1. **SQL:** Напиши запрос, который находит самый дорогой заказ для каждого пользователя.
2. **JOIN:** Найди всех пользователей, которые еще не сделали ни одного заказа.
3. **JSON:** Есть список `[{'meta': {'score': 10}}, {'meta': {'score': 20}}]`. Преврати его в DataFrame с колонкой `meta.score`.

---
## 🎯 Реальные вопросы с собеседований (Data Engineer / Data Analyst)

### 🎤 Теоретические вопросы:
1. **JOIN vs ON vs WHERE:** В чем фундаментальное отличие условий, переданных в блок `ON` во время `LEFT JOIN`, от тех же условий, переданных в блоке `WHERE`?
2. **UNION vs UNION ALL:** Что работает быстрее и почему? Какую операцию неявно делает `UNION` под капотом?
3. **Индексы:** Что такое индексы в БД (B-Tree). В каких случаях добавление индекса на колонку **замедлит** работу базы данных?

### 💻 Классические задачи на SQL (писать на доске / в блокноте):
1. **Топ-3 в группе (Хардкор):** Есть таблица `Employees(id, dept_id, salary)`. Напишите запрос, который выведет топ-3 самых высокооплачиваемых сотрудников в *каждом* департаменте. (Подсказка: `DENSE_RANK() OVER(PARTITION BY dept_id ORDER BY salary DESC)`).
2. **Удержание (Retention):** Как посчитать Retention 1-го дня? У вас есть таблица логов заходов `logins(user_id, login_date)`. (Подсказка: `LEFT JOIN` таблицы самой с собой со сдвигом даты `date + interval 1 day`).

### ✅ Ответы на вопросы

<details>
<summary><b>Ответ 1: JOIN ON vs WHERE</b></summary>

- Условие в **ON** определяет, *как* таблицы соединяются. При `LEFT JOIN` все строки из левой таблицы останутся в результате в любом случае (если для них нет совпадений по условию ON, они получат NULL).
- Условие в **WHERE** фильтрует *уже соединенный* результат. Если вы при `LEFT JOIN` напишете условие фильтрации правой таблицы в `WHERE`, вы фактически превратите его в `INNER JOIN`, так как строки с NULL отсекутся.
</details>

<details>
<summary><b>Ответ 2: UNION vs UNION ALL</b></summary>

**`UNION ALL` работает быстрее.**
- `UNION` объединяет результаты двух запросов, но затем неявно делает тяжелую операцию `DISTINCT` — убирает дубликаты.
- `UNION ALL` просто "склеивает" два результата вместе (даже если есть дубликаты), что не требует дополнительной сортировки и проверки уникальности.
</details>

<details>
<summary><b>Ответ 3: Индексы (B-Tree) и замедление работы</b></summary>

**Индексы** (обычно построенные на основе B-Tree) ускоряют чтение (`SELECT`).
**Когда замедляют:** Они серьезно замедляют операции записи (`INSERT`, `UPDATE`, `DELETE`), так как БД при каждом изменении должна не только перестроить таблицу, но и обновить структуру индекса. Поэтому нельзя вешать индексы на все колонки.
</details>

<details>
<summary><b>Ответ 4: Топ-3 (Оконные функции) и Retention</b></summary>

**Топ-3 в группе:**
```sql
WITH Ranked AS (
  SELECT *, DENSE_RANK() OVER(PARTITION BY dept_id ORDER BY salary DESC) as rnk
  FROM Employees
)
SELECT * FROM Ranked WHERE rnk <= 3;
```
**Retention 1-го дня:**
```sql
SELECT COUNT(DISTINCT b.user_id) * 100.0 / COUNT(DISTINCT a.user_id) as retention_day_1
FROM logins a
LEFT JOIN logins b ON a.user_id = b.user_id AND DATE(b.login_date) = DATE(a.login_date, '+1 day')
```
</details>

---
## 🧠 Частые дополнительные вопросы (на понимание)

<details>
<summary><b>1. Свойства ACID в базах данных</b></summary>

- **A (Atomicity):** Атомарность — транзакция выполняется либо полностью, либо никак.
- **C (Consistency):** Консистентность — транзакция переводит БД из одного валидного состояния в другое.
- **I (Isolation):** Изолированность — параллельные транзакции не влияют друг на друга.
- **D (Durability):** Долговечность — успешная транзакция сохраняется навсегда даже при сбоях питания.
</details>

<details>
<summary><b>2. Разница между OLTP и OLAP?</b></summary>

- **OLTP (Online Transaction Processing):** Обычные БД (PostgreSQL, MySQL). Заточены под быстрые вставки/изменения коротких записей (покупки, корзины пользователя). Данные сильно нормализованы.
- **OLAP (Online Analytical Processing):** Хранилища данных (ClickHouse, Snowflake, Greenplum). Заточены под тяжелые аналитические `SELECT` по миллиардам строк. Часто используют столбцовое хранение. Данные денормализованы.
</details>